In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from dataSet import SGNS_store_DataSet, dataset_weighted

from typing import Sequence, Optional, Callable, List, Dict

from copy import deepcopy

import nltk
from nltk.tokenize import word_tokenize
# nltk.download('punkt_tab') # A faire la première fois

import seaborn as sns
import matplotlib.pyplot as plt

import unicodedata
import string

from visuEmbedding import components_to_fig_3D, components_to_fig_3D_animation, components_to_fig_3D_simple
import plotly.express as px
import plotly.graph_objects as go

import tool
from data.pipData import pipe_data, prepare_data, prepare_data_with_intonation, separate_text_intonation

import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import skew

from collections import Counter

from data.pipData import separate_text_intonation
from dataSet import W2V_weighted_DataSet, W2V_weighted_DataSet_v2

[nltk_data] Downloading package punkt_tab to /home/pe/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/pe/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


# Fonction and class
class :
`SGNS_OneEmbWeighted`
`SGNS_Weighted`
`SGNS_WeightedHooktNeg`

fct :
`compute_importance`

In [2]:
class SGNS_OneEmbWeighted(nn.Module):
    """
    Use only one embedding and apply a weight in loss :
    loss = -((pos_loss + neg_loss) * weights).mean()
    """
    def __init__(self, emb_size:int, embedding_dimension:int=15, init_range:float|None=None, 
                sparse:bool=True, device="cpu"):
        super().__init__()
        self.emb_size:int = emb_size
        self.emb_dim:int = embedding_dimension
        self.word_emb:nn.Embedding = nn.Embedding(num_embeddings=self.emb_size,
                                                embedding_dim=self.emb_dim, device=device, sparse=sparse)

        if init_range is None:
            init_range = 0.5 / self.emb_dim
        self.init_range:float = init_range
        self.word_emb.weight.data.uniform_(-init_range, init_range)

    def forward(self, data:tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]):
        """
        Forward for model using a weight in its loss
        
        :param data: tuple contain centrals words, positive context, negative context, weights
        :type data: tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]
        """
        centrals_words, pos_context, neg_context, weights = data
        words_emb:torch.Tensor = self.word_emb(centrals_words) # [B, D]
        context_emb:torch.Tensor = self.word_emb(pos_context) # [B, D]
        neg_emb:torch.Tensor = self.word_emb(neg_context) # [B, K, D]

        pos_score = torch.sum(words_emb * context_emb, dim=1)
        pos_loss = F.logsigmoid(pos_score)

        neg_score = torch.bmm(neg_emb, words_emb.unsqueeze(-1)).squeeze(2)
        neg_loss = F.logsigmoid(-neg_score).sum(1)
        loss = -((pos_loss + neg_loss) * weights).mean()
        
        return loss

In [3]:
class SGNS_Weighted(nn.Module):
    """
    Apply a weight in loss :
    loss = -((pos_loss + neg_loss) * weights).mean()
    """
    def __init__(self, emb_size:int, embedding_dimension:int=15, init_range:float|None=None, 
                sparse:bool=True, device="cpu"):
        super().__init__()
        self.emb_size:int = emb_size
        self.emb_dim:int = embedding_dimension
        self.word_emb:nn.Embedding = nn.Embedding(num_embeddings=self.emb_size,
                                                embedding_dim=self.emb_dim, device=device, 
                                                sparse=sparse)
        self.con_emb:nn.Embedding = nn.Embedding(num_embeddings=self.emb_size, embedding_dim=self.emb_dim, 
                                                device=device,sparse=sparse)

        if init_range is None:
            init_range = 0.5 / self.emb_dim
        self.init_range:float = init_range
        self.word_emb.weight.data.uniform_(-init_range, init_range)
        self.con_emb.weight.data.uniform_(-init_range, init_range)

    def forward(self, data:tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]):
        """
        Forward for model using a weight in its loss
        
        :param data: tuple contain centrals words, positive context, negative context, weights
        :type data: tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]
        """
        centrals_words, pos_context, neg_context, weights = data
        words_emb:torch.Tensor = self.word_emb(centrals_words) # [B, D]
        context_emb:torch.Tensor = self.con_emb(pos_context) # [B, D]
        neg_emb:torch.Tensor = self.con_emb(neg_context) # [B, K, D]

        pos_score = torch.sum(words_emb * context_emb, dim=1)
        pos_loss = F.logsigmoid(pos_score)

        neg_score = torch.bmm(neg_emb, words_emb.unsqueeze(-1)).squeeze(2)
        neg_loss = F.logsigmoid(-neg_score).sum(1)
        loss = -((pos_loss + neg_loss) * weights).mean()
        
        return loss

In [4]:
class SGNS_WeightedHooktNeg(nn.Module):
    """
    Apply a weight in loss :
    loss = -((pos_loss + neg_loss) * weights).mean()
    """
    def __init__(self, emb_size:int, embedding_dimension:int=15, init_range:float|None=None, 
                sparse:bool=True, device="cpu"):
        super().__init__()
        self.emb_size:int = emb_size
        self.emb_dim:int = embedding_dimension
        self.word_emb:nn.Embedding = nn.Embedding(num_embeddings=self.emb_size,
                                                embedding_dim=self.emb_dim, device=device, 
                                                sparse=sparse)
        self.con_emb:nn.Embedding = nn.Embedding(num_embeddings=self.emb_size, embedding_dim=self.emb_dim, 
                                                device=device,sparse=sparse)

        if init_range is None:
            init_range = 0.5 / self.emb_dim
        self.init_range:float = init_range
        self.word_emb.weight.data.uniform_(-init_range, init_range)
        self.con_emb.weight.data.uniform_(-init_range, init_range)

    def forward(self, data:tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]):
        """
        Forward for model using a weight in its loss
        
        :param data: tuple contain centrals words, positive context, negative context, weights
        :type data: tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]
        """
        centrals_words, pos_context, neg_context, weights = data
        words_emb:torch.Tensor = self.word_emb(centrals_words) # [B, D]
        context_emb:torch.Tensor = self.con_emb(pos_context) # [B, D]
        neg_emb:torch.Tensor = self.con_emb(neg_context) # [B, K, D]
        
        def weight_neg_hook(grad):
            return grad * 0.1
        
        neg_emb.register_hook(weight_neg_hook)

        pos_score = torch.sum(words_emb * context_emb, dim=1)
        pos_loss = F.logsigmoid(pos_score)

        neg_score = torch.bmm(neg_emb, words_emb.unsqueeze(-1)).squeeze(2)
        neg_loss = F.logsigmoid(-neg_score).sum(1)
        loss = -((pos_loss + neg_loss) * weights).mean()
        
        return loss

In [5]:
def compute_importance(words, intonations):
        dict_list_importance = {}
        for sentence, intonation in zip(words, intonations) :
            for index, inton in enumerate(intonation):
                if sentence[index] not in dict_list_importance :
                    dict_list_importance[sentence[index]] = [float(inton)]
                else :
                    dict_list_importance[sentence[index]].append(float(inton))

        dict_importance = {}
        for word in dict_list_importance :
            dict_importance[word] = sum(dict_list_importance[word]) / len(dict_list_importance[word])

        return dict_importance

# Data
Create texts and intonation

## GNG

In [ ]:
data = prepare_data_with_intonation(
    file_path="./data/GoodNightGorilla_Intonation.txt",
    language='english',
    remove_accent=True,
    remove_punct=True,
    keep_apostrophes=False,
    contraction_map={# specific to corpus 
        "that's" : "thatis",
        "it's" : "itis",
        "don't": "donot",
        "doesn't": "doesnot",},
    stop_words=["s", "n't"],
    break_line=False
)
texts, intonations = separate_text_intonation(data)
importance = compute_importance(texts, intonations)
bins = [0, 1, 2, 3, 4, 5]
labels = ['0-1', '1-2', '2-3', '3-4', '4-5']
df_of_importance = pd.DataFrame(importance.values(), index=importance.keys())
df_of_importance['range'] = pd.cut(df_of_importance[0], bins=bins, labels=labels, include_lowest=True)
proportions = df_of_importance['range'].value_counts(normalize=True).sort_index(ascending=True)

## IWW

In [ ]:
data = prepare_data_with_intonation(
    file_path="data/IwentWalking_intonation.txt",
    language='english',
    remove_accent=True,
    remove_punct=True,
    keep_apostrophes=False,
    contraction_map={
        "that's" : "thatis",
        "it's" : "itis",
        "don't": "donot",
        "doesn't": "doesnot",
        "you're": "youre"
    },
    stop_words=["s", "n't"],
    break_line=False
)
texts, intonations = separate_text_intonation(data)
importance = compute_importance(texts, intonations)
bins = [0, 1, 2, 3, 4, 5]
labels = ['0-1', '1-2', '2-3', '3-4', '4-5']
df_of_importance = pd.DataFrame(importance.values(), index=importance.keys())
df_of_importance['range'] = pd.cut(df_of_importance[0], bins=bins, labels=labels, include_lowest=True)
proportions = df_of_importance['range'].value_counts(normalize=True).sort_index(ascending=True)

## GNG and IWW

In [ ]:
data = prepare_data_with_intonation(
    file_path="./data/GoodNightGorilla_Intonation.txt",
    language='english',
    remove_accent=True,
    remove_punct=True,
    keep_apostrophes=False,
    contraction_map={# specific to corpus 
        "that's" : "thatis",
        "it's" : "itis",
        "don't": "donot",
        "doesn't": "doesnot",},
    stop_words=["s", "n't"],
    break_line=False
)
texts, intonations = separate_text_intonation(data)

data = prepare_data_with_intonation(
    file_path="data/IwentWalking_intonation.txt",
    language='english',
    remove_accent=True,
    remove_punct=True,
    keep_apostrophes=False,
    contraction_map={
        "that's" : "thatis",
        "it's" : "itis",
        "don't": "donot",
        "doesn't": "doesnot",
        "you're": "youre"
    },
    stop_words=["s", "n't"],
    break_line=False
)
texts2, intonations2 = separate_text_intonation(data)
intonations.extend(intonations2)
texts.extend(texts2)
importance = compute_importance(texts, intonations)
bins = [0, 1, 2, 3, 4, 5]
labels = ['0-1', '1-2', '2-3', '3-4', '4-5']
df_of_importance = pd.DataFrame(importance.values(), index=importance.keys())
df_of_importance['range'] = pd.cut(df_of_importance[0], bins=bins, labels=labels, include_lowest=True)
proportions = df_of_importance['range'].value_counts(normalize=True).sort_index(ascending=True)

## TS

In [ ]:
data = prepare_data_with_intonation(
    file_path="data/TheSnowman_intonation.txt",
    language='english',
    remove_accent=True,
    remove_punct=True,
    keep_apostrophes=False,
    contraction_map={
        "that's" : "thatis",
        "it's" : "itis",
        "don't": "donot",
        "doesn't": "doesnot",
        "you're": "youre"},
    stop_words=["s", "n't"],
    break_line=False
)
texts, intonations = separate_text_intonation(data)   
importance = compute_importance(texts, intonations)
print(importance)
bins = [0, 1, 2, 3, 4, 5]
labels = ['0-1', '1-2', '2-3', '3-4', '4-5']
df_of_importance = pd.DataFrame(importance.values(), index=importance.keys())
df_of_importance['range'] = pd.cut(df_of_importance[0], bins=bins, labels=labels, include_lowest=True)
proportions = df_of_importance['range'].value_counts(normalize=True).sort_index(ascending=True)

# TS + GNG + IWW

In [6]:
data = prepare_data_with_intonation(
    file_path="./data/GoodNightGorilla_Intonation.txt",
    language='english',
    remove_accent=True,
    remove_punct=True,
    keep_apostrophes=False,
    contraction_map={# specific to corpus 
        "that's" : "thatis",
        "it's" : "itis",
        "don't": "donot",
        "doesn't": "doesnot",},
    stop_words=["s", "n't"],
    break_line=False
)
texts, intonations = separate_text_intonation(data)

data = prepare_data_with_intonation(
    file_path="data/IwentWalking_intonation.txt",
    language='english',
    remove_accent=True,
    remove_punct=True,
    keep_apostrophes=False,
    contraction_map={
        "that's" : "thatis",
        "it's" : "itis",
        "don't": "donot",
        "doesn't": "doesnot",
        "you're": "youre"
    },
    stop_words=["s", "n't"],
    break_line=False
)
texts2, intonations2 = separate_text_intonation(data)
intonations.extend(intonations2)
texts.extend(texts2)

data = prepare_data_with_intonation(
    file_path="data/TheSnowman_intonation.txt",
    language='english',
    remove_accent=True,
    remove_punct=True,
    keep_apostrophes=False,
    contraction_map={
        "that's" : "thatis",
        "it's" : "itis",
        "don't": "donot",
        "doesn't": "doesnot",
        "you're": "youre"
    },
    stop_words=["s", "n't"],
    break_line=False
)
texts2, intonations2 = separate_text_intonation(data)
intonations.extend(intonations2)
texts.extend(texts2)
importance = compute_importance(texts, intonations)
bins = [0, 1, 2, 3, 4, 5]
labels = ['0-1', '1-2', '2-3', '3-4', '4-5']
df_of_importance = pd.DataFrame(importance.values(), index=importance.keys())
df_of_importance['range'] = pd.cut(df_of_importance[0], bins=bins, labels=labels, include_lowest=True)
proportions = df_of_importance['range'].value_counts(normalize=True).sort_index(ascending=True)

# Normalisation and data set / loader (bs = 16)
norm01 : range_norm = 1.9 and center_norm = 1.  
norm02 : range_norm = 1.75 and center_norm = 1.  
norm03 : range_norm = 1.5 and center_norm = 1.


In [7]:
range_norm = 1.9
center_norm = 1.
intonations_normalize = tool.normalize_range_center(intonations, range_normalize=range_norm, center=center_norm)

dataset:dataset_weighted = dataset_weighted(sentences=texts,
                                intonations=intonations_normalize, nb_neg=10, window_size=6)
loader = DataLoader(dataset, batch_size=16, shuffle=True)


# Model

## One Emb

In [8]:
dim = 3
modelOneEmb = SGNS_OneEmbWeighted(emb_size=dataset.vocab_size,
                            embedding_dimension=dim, init_range=None, device="cuda", sparse=False)
optimizer = torch.optim.Adam(modelOneEmb.parameters(), lr=0.003)

w = deepcopy(modelOneEmb.word_emb.weight.detach().cpu().numpy())
emb_by_epoch = [w]

nb_epoch = 15
loss_by_epoch:list[float] = []
for epoch in range(nb_epoch):
    loss_in_epoch:list[float] = []
    for sentence_nb, data in enumerate(loader):
        data = [d.to("cuda") for d in data]
        optimizer.zero_grad()
        loss:torch.Tensor = modelOneEmb(data)
        loss.backward()
        optimizer.step()
        loss_in_epoch.append(loss.detach().cpu().item())
    loss_by_epoch.append(np.mean(loss_in_epoch))
    w = deepcopy(modelOneEmb.word_emb.weight.detach().cpu().numpy())
    emb_by_epoch.append(w)
oneEmb_hist = emb_by_epoch
if dim > 3:
    print('apply PCA in embedding')
    histo_emb_norm = [] # Normalisation PCA for embedding > 3
    for emb in emb_by_epoch:
        pca = PCA(n_components=3)
        X = pca.fit_transform(emb)
        histo_emb_norm.append(X)
        
    oneEmb_hist = histo_emb_norm

## Two Emb

In [ ]:
dim = 3
modelTwoEmb = SGNS_Weighted(emb_size=dataset.vocab_size,
                            embedding_dimension=dim, init_range=None, device="cuda", sparse=False)
optimizer = torch.optim.Adam(modelTwoEmb.parameters(), lr=0.003)

w = deepcopy(modelTwoEmb.word_emb.weight.detach().cpu().numpy())
c = deepcopy(modelTwoEmb.con_emb.weight.detach().cpu().numpy())
emb_by_epoch_word = [w]
emb_by_epoch_context = [c]

nb_epoch = 25
loss_by_epoch:list[float] = []
for epoch in range(nb_epoch):
    loss_in_epoch:list[float] = []
    for sentence_nb, data in enumerate(loader):
        data = [d.to("cuda") for d in data]
        optimizer.zero_grad()
        loss:torch.Tensor = modelTwoEmb(data)
        loss.backward()
        optimizer.step()
        loss_in_epoch.append(loss.detach().cpu().item())
    loss_by_epoch.append(np.mean(loss_in_epoch))
    w = deepcopy(modelTwoEmb.word_emb.weight.detach().cpu().numpy())
    c = deepcopy(modelTwoEmb.con_emb.weight.detach().cpu().numpy())
    emb_by_epoch_word.append(w)
    emb_by_epoch_context.append(c)

twoEmb_hist = emb_by_epoch_word
twoEmb_hist_context = emb_by_epoch_context
twoEmb_hist_merge = []

if dim > 3:
    histo_emb_norm = [] # Normalisation PCA for embedding > 3
    for emb in emb_by_epoch:
        pca = PCA(n_components=3)
        X = pca.fit_transform(emb)
        histo_emb_norm.append(X)
        
    twoEmb_hist = histo_emb_norm   

## Two embedding but neg less weighted

In [ ]:
dim = 3
modelTwoEmbNeg = SGNS_WeightedHooktNeg(emb_size=dataset.vocab_size,
                            embedding_dimension=dim, init_range=None, device="cuda", sparse=False)
optimizer = torch.optim.Adam(modelTwoEmbNeg.parameters(), lr=0.003)

w = deepcopy(modelTwoEmbNeg.word_emb.weight.detach().cpu().numpy())
c = deepcopy(modelTwoEmbNeg.con_emb.weight.detach().cpu().numpy())
emb_by_epoch_word = [w]
emb_by_epoch_context = [c]

nb_epoch = 25
loss_by_epoch:list[float] = []
for epoch in range(nb_epoch):
    loss_in_epoch:list[float] = []
    for sentence_nb, data in enumerate(loader):
        data = [d.to("cuda") for d in data]
        optimizer.zero_grad()
        loss:torch.Tensor = modelTwoEmbNeg(data)
        loss.backward()
        optimizer.step()
        loss_in_epoch.append(loss.detach().cpu().item())
    loss_by_epoch.append(np.mean(loss_in_epoch))
    w = deepcopy(modelTwoEmbNeg.word_emb.weight.detach().cpu().numpy())
    c = deepcopy(modelTwoEmbNeg.con_emb.weight.detach().cpu().numpy())
    emb_by_epoch_word.append(w)
    emb_by_epoch_context.append(c)

twoEmb_hist_neg = emb_by_epoch_word
twoEmb_hist_neg_context = emb_by_epoch_context
twoEmb_hist_neg_merge = []

if dim > 3:
    histo_emb_norm = [] # Normalisation PCA for embedding > 3
    for emb in emb_by_epoch:
        pca = PCA(n_components=3)
        X = pca.fit_transform(emb)
        histo_emb_norm.append(X)
        
    twoEmb_hist_neg = histo_emb_norm

# Visualisation

In [ ]:
cluster_animal = ["lion", "gorilla", "mouse", "elephant",  "giraffe", "hyena", "armadillo"]
cluster_nature_TS = ["snow", "snowman", "snowball", "ice", "sun", "moon", "sky", "star"]
cluster_clothing_TS = ["hat", "scarf", "boots", "pajamas", "robe"]
cluster_house_TS = ["window", "door", "fireplace", "tv", "lamp", "refrigerator", "stove", "sink"]

base_colors_neighbor = {}
base_colors = {}
for w in cluster_animal:
    base_colors_neighbor[w] = ("red", "orange")
    base_colors[w] = "red"
    
base_colors_importance = {}
base_colors_bad_word = {}
cluster_bad_word = []
base_colors_importance_neighbor = {}
for t in df_of_importance["range"].to_dict().items():
    if t[1] == '0-1':
        base_colors_importance[t[0]] = "black"
        base_colors_importance_neighbor[t[0]] = ("black", "black")
        base_colors_bad_word[t[0]] = "black"
        cluster_bad_word.append(t[0])
    if t[1] == '1-2':
        base_colors_importance[t[0]] = "orange"
        base_colors_importance_neighbor[t[0]] = ("orange", "black")
    if t[1] == '2-3':
        base_colors_importance[t[0]] = "blue"
        base_colors_importance_neighbor[t[0]] = ("blue", "black")
    if t[1] == '3-4':
        base_colors_importance[t[0]] = "green"
        base_colors_importance_neighbor[t[0]] = ("green", "black")
    if t[1] == '4-5':
        base_colors_importance[t[0]] = "red"
        base_colors_importance_neighbor[t[0]] = ("red", "black")
        
cluster_IWW = ["dog", "pig", "horse", "cow",  "cat", "duck"]
base_colors_two_corpus_aimal = {}
for c in cluster_animal:
    base_colors_two_corpus_aimal[c] = "blue"
for c in cluster_IWW:
    base_colors_two_corpus_aimal[c] = "green"
    


data_intonation_IWW:pd.DataFrame = pd.read_csv("data/intonation_word_IWW.csv", index_col=0)
data_intonation_GNG:pd.DataFrame = pd.read_csv("data/intonation_word_GNG.csv", index_col=0)

word_in_commun = data_intonation_IWW.index.intersection(data_intonation_GNG.index)
word_only_IWW = data_intonation_IWW.index.difference(word_in_commun)
word_only_GNG = data_intonation_GNG.index.difference(word_in_commun)
base_colors_two_corpus = {}

for w in word_in_commun:
    base_colors_two_corpus[w] = "orange"
    
for w in word_only_IWW:
    base_colors_two_corpus[w] = "green"
    
for w in word_only_GNG:
    base_colors_two_corpus[w] = "blue"
    
base_colors_TS = {}
for w in cluster_clothing_TS:
    base_colors_TS[w] = "orange"
for w in cluster_nature_TS:
    base_colors_TS[w] = "green"
for w in cluster_house_TS:
    base_colors_TS[w] = "blue"

# Figure

## TS see different cluster

In [1]:
fig = components_to_fig_3D_simple(
    components=oneEmb_hist[-1],
    encoder=dataset.encoder,
    base_color=base_colors_TS,
    default_color="gray",
    default_opacity=0.3
)
fig.show()

# fig = components_to_fig_3D_simple(
#     components=twoEmb_hist[-1],
#     encoder=dataset.encoder,
#     base_color=base_colors_TS
# )
# fig.show()

NameError: name 'components_to_fig_3D_simple' is not defined

## Other

In [16]:
fig = components_to_fig_3D_animation(
    history_components=oneEmb_hist,
    encoder=dataset.encoder,
    highlight_words=cluster_animal,
    nb_neighbors=1, base_color=base_colors_neighbor
)

fig = components_to_fig_3D_simple(
    components=oneEmb_hist[-1],
    encoder=dataset.encoder,
    base_color=base_colors_two_corpus_aimal
)
custom_legend = {
    "Animaux présent dans GNG": "blue",
    "Animaux présent dans IWW": "green",
    "Autres mots": "lightgray"
}

fig.update_traces(selector=dict(name="Vectors"), showlegend=False)
for label, color in custom_legend.items():
    fig.add_trace(go.Scatter3d(
        x=[None],  # Aucune coordonnée = rien n'est dessiné
        y=[None],
        z=[None],
        mode='markers',
        marker=dict(size=10, color=color),  # Taille du point dans la légende
        name=label
    ))
    
fig.update_legends()
fig.show()
fig.show()

fig = components_to_fig_3D_simple(
    components=oneEmb_hist[-1],
    encoder=dataset.encoder,
    base_color=base_colors_two_corpus
)

custom_legend = {
    "Mots présent dans GNG": "blue",
    "Mots présent dans IWW": "green",
    "Mots présent dans les deux corpus": "orange"
}


fig.update_traces(selector=dict(name="Vectors"), showlegend=False)
for label, color in custom_legend.items():
    fig.add_trace(go.Scatter3d(
        x=[None],  # Aucune coordonnée = rien n'est dessiné
        y=[None],
        z=[None],
        mode='markers',
        marker=dict(size=10, color=color),  # Taille du point dans la légende
        name=label
    ))
    
fig.update_legends()
fig.show()

In [ ]:
fig = components_to_fig_3D_simple(
    components=oneEmb_hist[-1],
    encoder=dataset.encoder,
    base_color=base_colors_importance
)
fig.show()

fig = components_to_fig_3D_simple(
    components=twoEmb_hist[-1],
    encoder=dataset.encoder,
    base_color=base_colors_two_corpus_aimal
)

custom_legend = {
    "Animaux présent dans GNG": "blue",
    "Animaux présent dans IWW": "green",
    "Autres mots": "lightgray"
}

fig.update_traces(selector=dict(name="Vectors"), showlegend=False)
for label, color in custom_legend.items():
    fig.add_trace(go.Scatter3d(
        x=[None],  # Aucune coordonnée = rien n'est dessiné
        y=[None],
        z=[None],
        mode='markers',
        marker=dict(size=10, color=color),  # Taille du point dans la légende
        name=label
    ))
    
fig.update_legends()
fig.show()

fig = components_to_fig_3D_simple(
    components=twoEmb_hist[-1],
    encoder=dataset.encoder,
    base_color=base_colors_two_corpus
)

custom_legend = {
    "Mots présent dans GNG": "blue",
    "Mots présent dans IWW": "green",
    "Mots présent dans les deux corpus": "orange"
}


fig.update_traces(selector=dict(name="Vectors"), showlegend=False)
for label, color in custom_legend.items():
    fig.add_trace(go.Scatter3d(
        x=[None],  # Aucune coordonnée = rien n'est dessiné
        y=[None],
        z=[None],
        mode='markers',
        marker=dict(size=10, color=color),  # Taille du point dans la légende
        name=label
    ))
    
fig.update_legends()

In [ ]:
fig = components_to_fig_3D_animation(
    history_components=twoEmb_hist,
    encoder=dataset.encoder,
    highlight_words=cluster_bad_word,
    nb_neighbors=1, base_color=base_colors_neighbor
)
fig = components_to_fig_3D_animation(
    history_components=twoEmb_hist_context,
    encoder=dataset.encoder,
    highlight_words=cluster_bad_word,
    nb_neighbors=1, base_color=base_colors_neighbor
)

In [ ]:
fig = components_to_fig_3D_simple(
    components=twoEmb_hist[-1],
    encoder=dataset.encoder,
    base_color=base_colors_importance
)
fig.show()

In [ ]:
fig = components_to_fig_3D_simple(
    components=twoEmb_hist_context[-1],
    encoder=dataset.encoder,
    base_color=base_colors_importance
)
fig.show()

In [ ]:
fig = components_to_fig_3D_animation(
    history_components=twoEmb_hist_neg,
    encoder=dataset.encoder,
    highlight_words=cluster_bad_word,
    nb_neighbors=1
)

In [ ]:
fig = components_to_fig_3D_animation(
    history_components=twoEmb_hist_neg_context,
    encoder=dataset.encoder,
    highlight_words=cluster_bad_word,
    nb_neighbors=1
)